Create deterministic pseudo_UIDs, compute rolling/expanding transaction aggregates, extract cyclical time features from TransactionDT, and record interaction flags.  
No target encoding. Only feature mappings that do not use isFraud as an aggregate source.  
Input: data/interim/pruned_eda.parquet  
Output: data/processed/features_static.parquet

In [2]:
import os
import numpy as np
import pandas as pd
from src.utils import reduce_mem_usage

In [ ]:


INTERIM_PATH = "../data/interim/pruned_eda.parquet"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

df = pd.read_parquet(INTERIM_PATH)
#df = reduce_mem_usage(df)

Memory usage decreased to 774.48 MB (0.1% reduction)


In [5]:
# Create rolling windows for last 24h, for amount of transactiones per UID 
# IEEE-CIS anonymized user identity. Real systems track a user_id, we synthesize one using immutable or slowly-changing attributes
df['card_birthday'] = df['TransactionDTDays'] - df['D1']   # D1 is days after card registration, so we try to guess approx creation date of a card
df['Pseudo_UID'] = df['card1'].astype('str')+'-'+df['card2'].astype('str') +'-'+df['card3'].astype('str') +'-'+df['card4'].astype('str') +'-'+df['card5'].astype('str') +'-'+df['card6'].astype('str')+'-'+df['addr1'].astype('str') +'-'+df['addr2'].astype('str') + df['card_birthday'].astype('str')


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21632\3508039662.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['card_birthday'] = df['TransactionDTDays'] - df['D1']   # D1 is days after card registration, so we try to guess approx creation date of a card
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21632\3508039662.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Pseudo_UID'] = df['card1'].astype('str')+'-'+df['card2'].astype('str') +'-'+df['card3'].astype('str') +'-'+df['card4'].astype('str') +'-'+df['card5'].astype(

In [6]:
df['hour_of_day']=(df['TransactionDT'] // 3600) % 24
df['day_of_week']=(df['TransactionDT'] // (3600*24)) % 7

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21632\2182029119.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['hour_of_day']=(df['TransactionDT'] // 3600) % 24
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21632\2182029119.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['day_of_week']=(df['TransactionDT'] // (3600*24)) % 7


This script calculates rolling 24-hour historical transaction statistics (count and average amount) per user without data leakage (it looks strictly at past transactions, excluding the current one).Instead of using slow pandas operations like df.groupby(...).rolling('24h'), it converts the data into raw 1D NumPy arrays and uses binary search (np.searchsorted) and prefix sums (np.cumsum) to compute these rolling metrics at $O(N \log K)$ speed with minimal memory overhead.

In [ ]:
# Temporal Velocity Aggregations (Causal, Before Split)

# 1. Sort dataframe by UID and TransactionDT (in-place to save memory)
df.sort_values(['Pseudo_UID', 'datetime'], inplace=True)
df.reset_index(drop=True, inplace=True)

# 2. Extract minimal 1D arrays
uids = df['Pseudo_UID'].values
times = df['datetime'].values
amts = df['TransactionAmt'].values

n = len(df)
counts = np.zeros(n, dtype=np.int32)
means = np.full(n, np.nan, dtype=np.float32)

# 3. Find slice boundaries where Pseudo_UID changes
change_idx = np.flatnonzero(uids[:-1] != uids[1:]) + 1
group_starts = np.concatenate(([0], change_idx))
group_ends = np.concatenate((change_idx, [n]))

WINDOW_SEC = 24 * 3600  # 86,400 seconds

# 4. Fast C-level searchsorted per user
for start, end in zip(group_starts, group_ends):
    t_group = times[start:end]
    a_group = amts[start:end]
    group_len = end - start

    # Prefix sums for O(1) interval sum
    prefix_sums = np.concatenate(([0.0], np.cumsum(a_group)))

    # Find the earliest index in window for each transaction: [t - 86400, t)
    # searchsorted('left') on (t_group - WINDOW_SEC) gives the start index
    left_indices = np.searchsorted(t_group, t_group - WINDOW_SEC, side='left')
    current_indices = np.arange(group_len)

    # Number of prior transactions in [t - 24h, t)
    cnt = current_indices - left_indices
    counts[start:end] = cnt

    # Window sum = prefix[current] - prefix[left]
    window_sums = prefix_sums[current_indices] - prefix_sums[left_indices]

    # Avoid div by 0; where count > 0, compute mean
    has_history = cnt > 0
    means_slice = np.full(group_len, np.nan, dtype=np.float32)
    means_slice[has_history] = window_sums[has_history] / cnt[has_history]
    means[start:end] = means_slice

# 5. Direct 1D assignment (zero index copying)
df['uid_count_24h'] = counts     # count of transactions by the user in prior 24 h
df['uid_amt_mean_24h'] = np.where(np.isnan(means), amts, means)  # average transaction amount spent by the user in prior 24 h


# Clean up raw arrays
del uids, times, amts, counts, means, group_starts, group_ends

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21632\3077040495.py:34: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  left_indices = np.searchsorted(t_group, t_group - WINDOW_SEC, side='left')
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21632\3077040495.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['uid_count_24h'] = counts
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21632\3077040495.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.conc

In [8]:
# Time elapsed since previous transaction for this UID
df['uid_prev_dt'] = df.groupby('Pseudo_UID')['datetime'].shift(1)
df['uid_dt_diff'] = df['datetime'] - df['uid_prev_dt']  # NaNs here mean 1st transaction

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21632\3216060668.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['uid_prev_dt'] = df.groupby('Pseudo_UID')['datetime'].shift(1)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21632\3216060668.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['uid_dt_diff'] = df['datetime'] - df['uid_prev_dt']  # NaNs here mean 1st transaction


In [9]:
# drop columns TransactionID, datetime, uid_prev_dt
df.drop(columns=['TransactionID', 'uid_prev_dt', 'datetime'], inplace=True)

In [10]:
# Frequency Encoding for High-Cardinality Categoricals
freq_cols = [
    # Card details
    'card1', 'card2', 'card3', 'card5',
    
    # Location
    'addr1', 'addr2',
    
    # Email providers
    'P_emaildomain', 'R_emaildomain',
    
    # Device & Identity metadata
    'DeviceInfo', 'id_30', 'id_31', 'id_33',
    
    # Composite proxy
    'Pseudo_UID'
]

for col in freq_cols:
    if col in df.columns:
        freq = df[col].value_counts(dropna=True)
        df[f'{col}_freq'] = df[col].map(freq).astype(np.float32)

df = reduce_mem_usage(df)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21632\2233324321.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_freq'] = df[col].map(freq).astype(np.float32)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21632\2233324321.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_freq'] = df[col].map(freq).astype(np.float32)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21632\2233324321.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which 

Memory usage decreased to 837.60 MB (0.6% reduction)


In [11]:
# List of columns to convert to category type for LGBM
categorical_cols = [
    # Card & Location features
    'card1', 'card2', 'card3', 'card5', 
    'addr1', 'addr2',
    
    # Match flags (excluding target-encoded M4)
    'M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9',
    
    # Identity details
    'DeviceType', 'DeviceInfo',
    'id_12', 'id_13', 'id_14', 'id_15', 'id_16', 'id_17', 'id_18', 'id_19',
    'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27',
    'id_28', 'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35',
    'id_36', 'id_37', 'id_38',
    
    # Time groupings
    'hour', 'dayofweek'
]

# Convert only columns that are present in your feature matrix
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

In [ ]:
output_path = os.path.join(PROCESSED_DIR, "features_static.parquet")
df.to_parquet(output_path, index=False, engine="pyarrow")
print(f"Features successfully engineered and saved to {output_path}. Shape: {df.shape}")

Features successfully engineered and saved to ../data/processed\features_static.parquet. Shape: (590540, 321)


: 